<a href="https://colab.research.google.com/github/shreeyashbaran/Info-5731/blob/main/In_class_exercises_4_Text_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Data Preprocessing & Cleaning (Text)  
**Time:** 20 minutes  |  **Points:** 10  

## Instructions
- This is an individual in-class assignment.  
- Write your code **inside each answer cell**.  
- Print the required outputs.  
- Submit your GitHub/Colab link as instructed by the instructor.


You are given a small dataset of customer support messages as a **TAB-separated text file**:  
- `support_messages.txt`

You will download this file from **Canvas** and upload it to your **Google Colab** notebook.

**How to upload it to your Google Colab notebook?**

1. Download `support_messages.txt` from Canvas.
3. In **the left sidebar**, click the **Files** icon (folder).  
4. Click **Upload** and select `support_messages.txt`.

6. RightAfter uploading, the file will appear in the Colab file list on the left.

6. Right-click the file, copy its path, and paste it into the FILE_PATH variable in Q1.

7. Run Q1 to load the dataset.



> Important: Keep the file name exactly as `support_messages.txt`.


## Questions (Total = 10 points)

### Q1 (1 point) — Load the dataset
Load the TAB-separated file into a pandas DataFrame with columns: `id`, `message`.  
Print: **(a)** `df.shape`, **(b)** `df.head(3)`.

### Q2 (3 points) — Descriptive columns
Add these columns for each message and print the full DataFrame:
- `word_count`: number of words  
- `char_count`: number of characters  
- `num_count`: number of digits (0–9)  
- `upper_word_count`: number of ALL-CAPS words (e.g., `"WHY"`, `"DAMAGED"`)  

### Q3 (3 points) — Clean text
Build a `clean_text(text)` function and create a new column `clean` with these steps **in order**:
1) lowercase  
2) remove punctuation/symbols (keep letters/numbers/spaces)  
3) remove English stopwords (use **nltk** or **sklearn** list)  
4) remove extra spaces  

Print the **original** message and **clean** version for rows `id=1` and `id=4`.

### Q4 (2 points) — Regex extraction
Using RegEx, extract and create two new columns:
- `order_id`: first occurrence of pattern `ORD-####` (case-insensitive; `ord-1060` is valid)  
- `email`: first email address if present (otherwise `None`/`NaN`)  

Print: `id`, `order_id`, `email` for all rows.

### Q5 (1 point) — TF-IDF keywords
Using the `clean` column, compute **TF-IDF** for the messages and print the **top 5 keywords** with the highest **average TF-IDF** across documents.


In [1]:
# Setup (run this cell first)
import re
import pandas as pd
from google.colab import files
import io

## Q1 (1 point) — Answer below

In [2]:
# Q1 — ANSWER CELL
# 1. Trigger file upload
uploaded = files.upload()

# 2. Get the filename dynamically
file_name = list(uploaded.keys())[0]

# TODO: load the TAB-separated file into df
# Hint: pd.read_csv(FILE_PATH, sep="\t")
df = pd.read_csv(file_name, sep="\t")

# TODO: print df.shape and df.head(3)
df.shape
df.head(3)

Saving support_messages.txt to support_messages.txt


,id,message
0,1,Hi!! My ORDER is late :( Order# ORD-1042. Ema...
1,2,Refund please!!! I was charged 2 times... invo...
2,3,"Great service, thanks! arrived in 2 days :)"


## Q2 (3 points) — Answer below

In [5]:
# Q2 — ANSWER CELL
# TODO: create word_count, char_count, num_count, upper_word_count
# Hint for digits: df["message"].str.count(r"\d")
# Hint for ALL-CAPS words: count tokens where token.isupper()

# TODO: display/print the full DataFrame

# word_count
df['word_count'] = df['message'].str.split().str.len()

# char_count
df['char_count'] = df['message'].str.len()

# num_count
df['num_count'] = df['message'].str.count(r"\d")

# upper_word_count
df['upper_word_count'] = df['message'].apply(lambda x: sum(word.isupper() for word in str(x).split()))


print(df)
df

   id                                            message  word_count  \
0   1  Hi!! My ORDER is late :(  Order# ORD-1042. Ema...          12   
1   2  Refund please!!! I was charged 2 times... invo...          11   
2   3        Great service, thanks! arrived in 2 days :)           8   
3   4  WHY is my package DAMAGED??? tracking says del...           8   
4   5  Need to change address: 7421 Frankford Rd Apt ...          12   
5   6  Support ticket: ORD-1050. Call me at (469) 555...           9   
6   7  I can’t login— password reset link not working...          10   
7   8  Item missing from box. pls send replacement!! ...          10   

   char_count  num_count  upper_word_count  
0          73          4                 2  
1          71          9                 3  
2          43          1                 0  
3          55          0                 2  
4          67         13                 1  
5          58         14                 2  
6          76          0            

,id,message,word_count,char_count,num_count,upper_word_count
0,1,Hi!! My ORDER is late :( Order# ORD-1042. Ema...,12,73,4,2
1,2,Refund please!!! I was charged 2 times... invo...,11,71,9,3
2,3,"Great service, thanks! arrived in 2 days :)",8,43,1,0
3,4,WHY is my package DAMAGED??? tracking says del...,8,55,0,2
4,5,Need to change address: 7421 Frankford Rd Apt ...,12,67,13,1
5,6,Support ticket: ORD-1050. Call me at (469) 555...,9,58,14,2
6,7,I can’t login— password reset link not working...,10,76,0,1
7,8,Item missing from box. pls send replacement!! ...,10,72,4,0


## Q3 (3 points) — Answer below

In [6]:
# Q3 — ANSWER CELL
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOPWORDS = set(ENGLISH_STOP_WORDS)

def clean_text(text: str) -> str:
    # 1. Lowercase
    text = str(text).lower()

    # 2. Remove punctuation/symbols (keep letters/numbers/spaces)
    # The regex [^a-z0-9\s] matches everything EXCEPT a-z, 0-9, and whitespace (\s)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    # 3. Remove English stopwords
    # 4. Remove extra spaces
    # Using .split() automatically handles removing extra spaces and tabs,
    # making it easy to filter stopwords at the same time.
    words = [word for word in text.split() if word not in STOPWORDS]

    return ' '.join(words)

# TODO: create df["clean"] using clean_text
df["clean"] = df["message"].apply(clean_text)

# TODO: print original and clean for id=1 and id=4
for val_id in [1, 4]:
    # Check if the id exists to avoid errors, then print
    row = df[df['id'] == val_id]
    if not row.empty:
        print(f"--- ID {val_id} ---")
        print(f"Original: {row['message'].values[0]}")
        print(f"Cleaned:  {row['clean'].values[0]}\n")

--- ID 1 ---
Original: Hi!! My ORDER is late :(  Order# ORD-1042. Email me at sara.Ali@gmail.com
Cleaned:  hi order late order ord1042 email saraaligmailcom

--- ID 4 ---
Original: WHY is my package DAMAGED??? tracking says delivered...
Cleaned:  package damaged tracking says delivered



## Q4 (2 points) — Answer below

In [7]:

import re

# order_id pattern: r"ORD-\d{4}" with re.IGNORECASE
# Added () to create a capture group for pandas str.extract
order_pattern = r"(ORD-\d{4})"

# email pattern (simple): r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
# Added () to create a capture group for pandas str.extract
email_pattern = r"([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})"

# TODO: create df["order_id"] and df["email"]
df["order_id"] = df["message"].str.extract(order_pattern, flags=re.IGNORECASE)
df["email"] = df["message"].str.extract(email_pattern)

# TODO: print/display df[["id", "order_id", "email"]]
print(df[["id", "order_id", "email"]])

   id  order_id                  email
0   1  ORD-1042     sara.Ali@gmail.com
1   2  ORD-1042                    NaN
2   3       NaN                    NaN
3   4       NaN                    NaN
4   5       NaN                    NaN
5   6  ORD-1050                    NaN
6   7       NaN  mehri.sattari@unt.edu
7   8  ord-1060                    NaN


## Q5 (1 point) — Answer below

In [8]:
# Q5 — ANSWER CELL
# Hint: from sklearn.feature_extraction.text import TfidfVectorizer
# 1) fit TF-IDF on df["clean"]
# 2) compute average TF-IDF per term across documents
# 3) print top 5 terms + their average scores

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

# 1) fit TF-IDF on df["clean"]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["clean"])

# 2) compute average TF-IDF per term across documents
# .mean(axis=0) calculates the mean for each column (term).
# np.asarray().ravel() flattens it into a standard 1D array.
avg_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).ravel()

# Extract the vocabulary (words) to pair with the scores
feature_names = vectorizer.get_feature_names_out()

# Create a Pandas Series to easily map terms to their average scores
tfidf_series = pd.Series(avg_tfidf, index=feature_names)

# 3) print top 5 terms + their average scores
top_5_terms = tfidf_series.sort_values(ascending=False).head(5)

print("Top 5 terms by average TF-IDF:")
print(top_5_terms)


Top 5 terms by average TF-IDF:
order        0.113518
ord1042      0.082873
email        0.079468
delivered    0.055902
days         0.055902
dtype: float64


## Grading Checklist
- Q1: correct load + prints  
- Q2: correct counts  
- Q3: cleaning follows the required order + prints for id=1 and id=4  
- Q4: regex extraction works (case-insensitive `ORD-####` and emails)  
- Q5: prints 5 keywords + their scores (rounding is fine)
